In [28]:
using MultivariateStats
using Distributions
using PlotlyJS

# --- Data Generation ---

# Number of samples per class
n_samples = 100

# 3 Neurons' response means to four conditions
response_means = [
    [20.0, 18.0, 30.0],
    [2.0, 2.0, 10.0],
    [1.0, 28.0, 30.0],
    [20.0, 10.0, 12.0]
]

# Neurons' covariance matrix
covariance = [
    1.0 0.7 0.5;
    0.7 1.0 0.2;
    0.5 0.2 1;
]

# Generate neuron data for each condition
neuron_data = vcat(
    (rand(MvNormal(response_means[1], covariance), n_samples))',
    (rand(MvNormal(response_means[2], covariance), n_samples))',
    (rand(MvNormal(response_means[3], covariance), n_samples))',
    (rand(MvNormal(response_means[4], covariance), n_samples))'
)

println(size(neuron_data))

# Create traces for each neuron
traces = PlotlyJS.AbstractTrace[] 
for n in 1:size(neuron_data, 2)
    trace = scatter(y=neuron_data[:, n], name="Neuron $(n)")
    push!(traces, trace)
end

# Create the plot layout
layout = Layout(
    title="Neuron Responses",
    xaxis_title="trial#",
    yaxis_title="Response"
)

# Create the plot
plt = plot(traces, layout)

# Display the plot
display(plt)

(400, 3)


┌ Warning: It looks like the Kaleido process is not responding. 
│ The unresponsive process will be killed, but this means that you will not be able to save figures using `savefig`.
│ 
│ If you are on Windows this might be caused by known problems with Kaleido v0.2 on Windows (you are using version 0.2.1).
│ You might want to try forcing a downgrade of the Kaleido_jll library to 0.1.
│ Check the Package Readme at https://github.com/JuliaPlots/PlotlyKaleido.jl/tree/main#windows-note for more details.
│ 
│ If you think this is not your case, you might try using a longer timeout to check if the process is not responding (defaults to 10 seconds) by passing the desired value in seconds using the `timeout` kwarg when calling `PlotlyKaleido.start` or `PlotlyKaleido.restart`
└ @ PlotlyKaleido C:\Users\86181\.julia\packages\PlotlyKaleido\U5CX4\src\PlotlyKaleido.jl:24
┌ Warning: It looks like the Kaleido process is not responding. 
│ The unresponsive process will be killed, but this means that y

data: [
  "scatter with fields name, type, and y",
  "scatter with fields name, type, and y",
  "scatter with fields name, type, and y"
]

layout: "layout with fields margin, template, title, xaxis, and yaxis"

**2)**
Start by computing the covariance of the responses. You can do this using `cov()`, or by subtracting off the mean response from each neuron (each column of `neuron_data`) and computing $\frac{neuron_data^Tneuron_data}{n − 1}$. Compute the eigenvalues, $\lambda_k$ of the covariance matrix with `eigen()`, and plot them in descending order. Do the data points live close to a subspace of dimensionality less than four?

In [29]:
using LinearAlgebra
using Statistics

# Calculate covariance using built-in function
cov_1 = cov(neuron_data)

# Alternatively, Let's calculate cov ourselves
neuron_data_norm = neuron_data .- mean(neuron_data, dims=1)
n = size(neuron_data, 1)  # Number of observations
cov_2 = (neuron_data_norm' * neuron_data_norm) / (n - 1)  # Covariance matrix
# cov_2 = *************

# check if they are the same
display(cov_1)
display(cov_2)

3×3 Matrix{Float64}:
 87.6393   -7.37299   2.20152
 -7.37299  93.2979   83.1251
  2.20152  83.1251   92.2837

3×3 Matrix{Float64}:
 87.6393   -7.37299   2.20152
 -7.37299  93.2979   83.1251
  2.20152  83.1251   92.2837

In [30]:
# Compute eigenvalues and eigenvectors of the covariance matrix
eigenvalues, eigenvectors = eigen(cov_1)

# Julia's eigen() returns eigenvalues in ascending order, let's reverse them for descending order
eigenvalues = reverse(eigenvalues)
eigenvectors = eigenvectors[:, reverse(1:end)]


# Plot eigenvalues using PlotlyJS
plt = PlotlyJS.plot(
    [PlotlyJS.scatter(y=eigenvalues, mode="lines+markers", marker_color="red"), ],
    Layout(
        title="Eigenvalues of Covariance neuron_dataatrix",
        xaxis_title="λ_k"
        
    )
)

display(plt)


┌ Warning: It looks like the Kaleido process is not responding. 
│ The unresponsive process will be killed, but this means that you will not be able to save figures using `savefig`.
│ 
│ If you are on Windows this might be caused by known problems with Kaleido v0.2 on Windows (you are using version 0.2.1).
│ You might want to try forcing a downgrade of the Kaleido_jll library to 0.1.
│ Check the Package Readme at https://github.com/JuliaPlots/PlotlyKaleido.jl/tree/main#windows-note for more details.
│ 
│ If you think this is not your case, you might try using a longer timeout to check if the process is not responding (defaults to 10 seconds) by passing the desired value in seconds using the `timeout` kwarg when calling `PlotlyKaleido.start` or `PlotlyKaleido.restart`
└ @ PlotlyKaleido C:\Users\86181\.julia\packages\PlotlyKaleido\U5CX4\src\PlotlyKaleido.jl:24
┌ Warning: It looks like the Kaleido process is not responding. 
│ The unresponsive process will be killed, but this means that y

data: [
  "scatter with fields marker, mode, type, and y"
]

layout: "layout with fields margin, template, title, and xaxis"

**3)**
Project the data in `neuron_data` onto the first principal component (i.e., compute the inner product of the data vectors with the eigenvector corresponding to the maximal eigenvalue). Show that the sum of squares of these values (divided by $n − 1$) is equal to $\lambda_1$. What proportion of the total variability of the data (sum of squared data) does this component account for?

In [33]:
# Project neuron_data (with subtracted mean) onto the first eigenvector
neuron_data_centered = neuron_data .- mean(neuron_data, dims=1)
projection = neuron_data_centered * eigenvectors[:, 1]  # Project onto the first eigenvector 

# Calculate variance in the first eigenvector
var_1 = var(projection)
println("Variance in first eigenvector (calculated): ", var_1)
println("Variance in first eigenvector (from eigenvalue): ", eigenvalues[1])

# Calculate total variance
var_total = sum(var(neuron_data, dims=1))
println("Total variance (calculated): ", var_total)
println("Total variance (from eigenvalues): ", sum(eigenvalues))

# Calculate proportion of variance in the first component
println("Proportion of variance (calculated): ", var_1 / var_total)
println("Proportion of variance (from eigenvalues): ", eigenvalues[1] / sum(eigenvalues))



Variance in first eigenvector (calculated): 176.07076212615385
Variance in first eigenvector (from eigenvalue): 176.07076212615388
Total variance (calculated): 273.22083056022313
Total variance (from eigenvalues): 273.220830560223
Proportion of variance (calculated): 0.6444265679345574
Proportion of variance (from eigenvalues): 0.6444265679345579


**4)**
Show a scatter plot of the data projected onto the first two principal components (that is, plot the dot product of the data with the first component versus the dot product with the second component). Use `plt.plot()`, requesting circular plot symbols and no connecting lines; set the two axes to use equal scales with `plt.xlim()` and `plt.ylim()`.

In [34]:
# Project mean-subtracted neuron_data onto the first two eigenvectors
neuron_data_centered = neuron_data .- mean(neuron_data, dims=1)
projection_1 = neuron_data_centered * eigenvectors[:, 1]  # Project onto the first eigenvector
projection_2 = neuron_data_centered * eigenvectors[:, 2]  # Project onto the second eigenvector

# Create class labels (1, 2, 3)
labels = repeat(1:length(response_means), inner = n_samples)

# Create the scatter plot using PlotlyJS
plt = PlotlyJS.plot(
    [PlotlyJS.scatter(x=projection_1, y=projection_2, mode="markers",marker=attr(color=labels, colorscale="viridis"),text=labels)],
    Layout(
        title="Projection onto First Two Eigenvectors",
        xaxis_title="PC1",
        yaxis_title="PC2",
        xaxis_range=[-30, 30],
        yaxis_range=[-30, 30]
    )
)
display(plt)

┌ Warning: It looks like the Kaleido process is not responding. 
│ The unresponsive process will be killed, but this means that you will not be able to save figures using `savefig`.
│ 
│ If you are on Windows this might be caused by known problems with Kaleido v0.2 on Windows (you are using version 0.2.1).
│ You might want to try forcing a downgrade of the Kaleido_jll library to 0.1.
│ Check the Package Readme at https://github.com/JuliaPlots/PlotlyKaleido.jl/tree/main#windows-note for more details.
│ 
│ If you think this is not your case, you might try using a longer timeout to check if the process is not responding (defaults to 10 seconds) by passing the desired value in seconds using the `timeout` kwarg when calling `PlotlyKaleido.start` or `PlotlyKaleido.restart`
└ @ PlotlyKaleido C:\Users\86181\.julia\packages\PlotlyKaleido\U5CX4\src\PlotlyKaleido.jl:24
┌ Warning: It looks like the Kaleido process is not responding. 
│ The unresponsive process will be killed, but this means that y

data: [
  "scatter with fields marker, mode, text, type, x, and y"
]

layout: "layout with fields margin, template, title, xaxis, and yaxis"

**5)**
Show that the sum of the squared lengths of these projected vectors (divided by $n−1$) is equal to $\lambda_1 + \lambda_2$. What proportion of the total variability of the data do these two components account for?

In [40]:
sum_squared_lengths = sum(projection_1.^2) + sum(projection_2.^2)
println("Sum of squared lengths: ", sum_squared_lengths / (size(neuron_data, 1) - 1))

# calcultate λ_1 + λ_2
println("λ_1 + λ_2: ", eigenvalues[1] + eigenvalues[2])

# calculate the proportion of explained variance of the first 2 eigenvectors
println("Proportion of explained variance: ", (eigenvalues[1] + eigenvalues[2]) / sum(eigenvalues))


# # calculate the sum of the squared lengths of these projected vectors
# println()

# # calculate λ_1 + λ_2
# println(*************)

# # calculate the proportion of explained variance of the first 2 eigenvectors
# println(***************)

Sum of squared lengths: 264.13881554282614
λ_1 + λ_2: 264.13881554282614
Proportion of explained variance: 0.9667594341223005


**6)**
For comparison, make a scatter plot of the data projected onto the first and third principal components (again, make sure the axes have the same scaling).

In [41]:
# Project mean-subtracted data onto the 1st and 3rd eigenvectors
neuron_data_centered = neuron_data .- mean(neuron_data, dims=1)
projection_1 = neuron_data_centered * eigenvectors[:, 1]  # Project onto the 1st eigenvector
projection_3 = neuron_data_centered * eigenvectors[:, 3]  # Project onto the 3rd eigenvector

# Create the scatter plot using PlotlyJS
plt = PlotlyJS.plot(
    [PlotlyJS.scatter(x=projection_1, y=projection_3, mode="markers",marker=attr(color=labels, colorscale="viridis"),text=labels)],
    Layout(
        title="Projection onto 1st and 3rd Eigenvectors",
        xaxis_title="PC1",
        yaxis_title="PC3",
        xaxis_range=[-30, 30],
        yaxis_range=[-30, 30]
    )
)
display(plt)

┌ Warning: It looks like the Kaleido process is not responding. 
│ The unresponsive process will be killed, but this means that you will not be able to save figures using `savefig`.
│ 
│ If you are on Windows this might be caused by known problems with Kaleido v0.2 on Windows (you are using version 0.2.1).
│ You might want to try forcing a downgrade of the Kaleido_jll library to 0.1.
│ Check the Package Readme at https://github.com/JuliaPlots/PlotlyKaleido.jl/tree/main#windows-note for more details.
│ 
│ If you think this is not your case, you might try using a longer timeout to check if the process is not responding (defaults to 10 seconds) by passing the desired value in seconds using the `timeout` kwarg when calling `PlotlyKaleido.start` or `PlotlyKaleido.restart`
└ @ PlotlyKaleido C:\Users\86181\.julia\packages\PlotlyKaleido\U5CX4\src\PlotlyKaleido.jl:24
┌ Warning: It looks like the Kaleido process is not responding. 
│ The unresponsive process will be killed, but this means that y

data: [
  "scatter with fields marker, mode, text, type, x, and y"
]

layout: "layout with fields margin, template, title, xaxis, and yaxis"

**6)**
Finally, let's try a different dimenstion reduction algorithm: tSNE.

In [43]:
using TSne

# --- t-SNE ---

tsne_model = tsne(neuron_data_centered, 2, 50, 1000, 20) # Run t-SNE
# Create class labels (1, 2, 3)
labels = repeat(1:length(response_means), inner = n_samples)

# t-SNE Plot
tsne_plot = PlotlyJS.plot(
    [
        PlotlyJS.scatter(
            x=tsne_model[:, 1],
            y=tsne_model[:, 2],
            mode="markers",
            marker=attr(color=labels, colorscale="viridis"),
            text=labels,
            name="t-SNE"
        )
    ],
    Layout(
        title="t-SNE Projection",
        xaxis_title="Dimension 1",
        yaxis_title="Dimension 2"
    )
)

display(tsne_plot)

Computing t-SNE  15%|██████                              |  ETA: 0:00:01
Computing t-SNE 100%|████████████████████████████████████| Time: 0:00:00
  KL_divergence:  0.2952
┌ Warning: It looks like the Kaleido process is not responding. 
│ The unresponsive process will be killed, but this means that you will not be able to save figures using `savefig`.
│ 
│ If you are on Windows this might be caused by known problems with Kaleido v0.2 on Windows (you are using version 0.2.1).
│ You might want to try forcing a downgrade of the Kaleido_jll library to 0.1.
│ Check the Package Readme at https://github.com/JuliaPlots/PlotlyKaleido.jl/tree/main#windows-note for more details.
│ 
│ If you think this is not your case, you might try using a longer timeout to check if the process is not responding (defaults to 10 seconds) by passing the desired value in seconds using the `timeout` kwarg when calling `PlotlyKaleido.start` or `PlotlyKaleido.restart`
└ @ PlotlyKaleido C:\Users\86181\.julia\packages\Pl

data: [
  "scatter with fields marker, mode, name, text, type, x, and y"
]

layout: "layout with fields margin, template, title, xaxis, and yaxis"